###Модуль 3: Enterprise LangChain. Агенты, Инструменты и LCEL.
**Цель модуля:**
Изучить архитектуру масштабируемых систем на базе чистого **LangChain (LCEL)**. Мы разберем, как управлять инструментами через Pydantic-схемы, как строить семантические роутеры для экономии ресурсов, как динамически подгружать инструменты (Tool Retrieval) и как обеспечивать надежность (Reliability) через встроенные механизмы Fallback и Retry.

**Стек:**
- **Inference:** Ollama + `qwen2.5-coder:3b`
- **Orchestration:** LangChain Expression Language (LCEL)
- **Validation:** Pydantic v2

# Настройка Среды

### 0. Настройка инфраструктуры
Поднимаем локальный сервер Ollama и скачиваем модель Qwen 2.5. Убедитесь, что в Colab включен GPU (T4).

In [2]:
# 1. Устанавливаем системную зависимость (zstd) для установки Ollama
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Устанавливаем ядро стека 2026 года (LangGraph исключен)
!pip install -qU langchain-core langchain-ollama langchain-community chromadb sentence-transformers pydantic

# 3. Устанавливаем движок Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Запускаем сервер в фоне и качаем модель
import os
import subprocess
import time

print("Инициализация сервера Ollama...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
with open("ollama.log", "w") as f:
    subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=os.environ)

# Даем серверу чуть больше времени на старт
time.sleep(7)

print("Скачивание Qwen 2.5 Coder (3B)...")
!ollama pull qwen2.5-coder:3b
print("✅ Окружение готово к Enterprise-разработке!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,921 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,652 

# Pydantic Guardrails

### 1. Schema Engineering: Строгая типизация и Pydantic Guardrails (Слайды 6, 10)

В 2026 году **The Schema is the Prompt**. Мы не просим модель текстово "выведи JSON в таком-то формате". Мы передаем ей строгую Pydantic-схему через `with_structured_output`.
Особое внимание обращаем на типы `Literal` и `Field` — это бесплатный способ защититься от галлюцинаций аргументов (Argument Hallucinations).

In [3]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal
from langchain_ollama import ChatOllama

# Инициализируем модель с нулевой температурой
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# 1. Проектируем строгую схему. Literal запрещает модели выдумывать свои статусы.
class DatabaseQuery(BaseModel):
    """Схема для безопасного запроса метрик из базы данных."""
    metric_name: Literal["cpu_usage", "ram_usage", "disk_io"] = Field(
        ...,
        description="Название метрики. Выбери строго из доступных вариантов."
    )
    server_region: Literal["EU", "US", "RU"] = Field(
        ...,
        description="Регион сервера."
    )
    limit: int = Field(
        default=10,
        description="Количество записей. Не более 100.",
        le=100 # Pydantic Guardrail: limit must be <= 100
    )

# Привязываем схему к модели
structured_llm = llm.with_structured_output(DatabaseQuery)

### 📝 Задание 1: LCEL Цикл самокоррекции (Feedback Loop)
В реальной жизни модель может проигнорировать `Literal` или передать строку вместо числа.
Напишите функцию на чистом Python, которая делает запрос к `structured_llm`. Если возникает `ValidationError`, функция ловит её (except ValidationError) и отправляет текст ошибки обратно модели с требованием исправить JSON.

In [4]:
def generate_with_correction(user_query: str, max_retries: int = 3):
    messages =[
        ("system", "Ты DevOps-ассистент. Твоя задача извлечь параметры для SQL-запроса."),
        ("user", user_query)
    ]

    # TODO: Напишите цикл (for/while) для обработки Exception
    # В случае ошибки добавляйте в список messages два сообщения:
    # 1. ("assistant", "Invalid JSON")
    # 2. ("user", f"Ошибка валидации: {e}. Исправь данные под схему.")

    pass # Ваш код здесь

# ТЕСТ (Модель должна понять, что 'ОАЭ' и 'network' - невалидны, и попытаться адаптировать или использовать дефолты)
# print(generate_with_correction("Дай мне логи сети для серверов в ОАЭ, 500 строчек!"))

<details>
<summary><b>💡 Показать решение (Задание 1)</b></summary>

```python
def generate_with_correction(user_query: str, max_retries: int = 3):
    messages =[
        ("system", "Ты DevOps-ассистент. Извлеки параметры. Если регион или метрика не совпадают с Literal, подбери ближайший или верни EU / cpu_usage."),
        ("user", user_query)
    ]
    
    for attempt in range(max_retries):
        try:
            print(f"Попытка {attempt + 1}...")
            # Пытаемся получить структурированный ответ
            result = structured_llm.invoke(messages)
            print("✅ Успешно распарсено!")
            return result
        except Exception as e: # Ловим ошибки парсинга Pydantic
            error_msg = str(e)
            print(f"❌ Ошибка валидации:\n{error_msg[:150]}...")
            
            # Сохраняем историю ошибки для модели (Feedback Loop)
            messages.append(("assistant", "I generated an invalid output."))
            messages.append(("user", f"Твой ответ вызвал ошибку валидации Pydantic: {error_msg}. Исправь значения согласно схеме."))
            
    print("🚨 Достигнут лимит ретраев. Fallback на дефолтные значения.")
    return DatabaseQuery(metric_name="cpu_usage", server_region="EU", limit=10)

# Запуск
print(generate_with_correction("Дай мне логи сети (network) для серверов в ОАЭ, 500 строчек!"))
```
</details>

#LCEL Semantic Router

### 2. Маршрутизация интентов (Semantic Router) (Слайд 11)

Вместо того чтобы гнать все запросы через тяжелые RAG-цепочки, мы ставим дешевый классификатор на входе. В LCEL это элегантно решается с помощью `RunnableBranch` или кастомной `RunnableLambda`.

In [5]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. Быстрый классификатор намерений
class IntentRouter(BaseModel):
    intent: Literal["RAG", "ACTION", "CHITCHAT"] = Field(
        description="RAG: поиск информации в доках. ACTION: выполнение команд (перезагрузка, деплой). CHITCHAT: болтовня."
    )

router_llm = llm.with_structured_output(IntentRouter)

def extract_intent(state: dict) -> dict:
    query = state["query"]
    try:
        decision = router_llm.invoke(query)
        intent = decision.intent
    except:
        intent = "CHITCHAT" # Fallback
    print(f"[ROUTER] Намерение: {intent}")
    return {"intent": intent, "query": query}

# 2. Создаем три разные цепочки (Chains) для разных задач
rag_chain = (
    ChatPromptTemplate.from_template("📚 [RAG] Ищу в базе политик ответ на: {query}")
    | llm | StrOutputParser()
)

action_chain = (
    ChatPromptTemplate.from_template("🛠️ [ACTION] Подготавливаю bash-скрипт для: {query}")
    | llm | StrOutputParser()
)

chitchat_chain = (
    ChatPromptTemplate.from_template("💬 [CHITCHAT] Ответь коротко и вежливо на: {query}")
    | llm | StrOutputParser()
)

### 📝 Задание 2: Сборка LCEL-цепочки с маршрутизацией
Вам нужно собрать единую LCEL цепочку (enterprise_router_chain).
Она должна начинаться с `RunnableLambda(extract_intent)`, а затем использовать паттерн `if/elif` внутри другой `RunnableLambda` для вызова правильной цепочки (`rag_chain`, `action_chain` или `chitchat_chain`).

In [6]:
# TODO: Реализуйте функцию маршрутизации (route_to_chain) и соберите пайплайн.
def route_to_chain(state: dict):
    # Ваш код: проверьте state["intent"] и верните нужную _цепочку_ (не результат её работы, а сам объект цепочки)
    pass

# enterprise_router_chain = ...

<details>
<summary><b>💡 Показать решение (Задание 2)</b></summary>

```python
def route_to_chain(state: dict):
    intent = state["intent"]
    if intent == "RAG":
        return rag_chain
    elif intent == "ACTION":
        return action_chain
    else:
        return chitchat_chain

# Сборка финального LCEL пайплайна
# RunnableLambda(route_to_chain) вернет нужный Chain, и LCEL автоматически передаст в него state!
enterprise_router_chain = (
    RunnableLambda(extract_intent)
    | RunnableLambda(route_to_chain)
)

# Тесты
print("--- Тест 1 ---")
print(enterprise_router_chain.invoke({"query": "Как оформить отпуск по регламенту?"}))
print("\n--- Тест 2 ---")
print(enterprise_router_chain.invoke({"query": "Перезагрузи production сервер БД"}))
print("\n--- Тест 3 ---")
print(enterprise_router_chain.invoke({"query": "Как твои дела, железяка?"}))
```
</details>

# Dynamic Tool Retrieval
### 3. Tool Retrieval: Динамический контекст инструментов (Слайд 7)

Если у вас 100+ инструментов (Jira, Confluence, AWS, Git), вы не можете передать их все в `llm.bind_tools()`. Контекст переполнится, и модель начнет путать аргументы.
Решение — хранить описания инструментов в векторной БД и "подтягивать" только релевантные под запрос пользователя!

In [7]:
from langchain_core.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Создаем "Зоопарк" инструментов
@tool
def reset_password(user_id: str):
    """Сбрасывает пароль пользователя в Active Directory."""
    return f"Password reset for {user_id}"

@tool
def get_jira_ticket(ticket_id: str):
    """Получает статус задачи из Jira."""
    return f"Status of {ticket_id} is IN PROGRESS"

@tool
def restart_pod(pod_name: str):
    """Перезапускает pod в Kubernetes кластере."""
    return f"Pod {pod_name} restarted"

@tool
def calculate_bonus(employee_id: str):
    """Считает годовой бонус сотрудника."""
    return f"Bonus for {employee_id} is 5000"

all_tools =[reset_password, get_jira_ticket, restart_pod, calculate_bonus]

# 2. Индексируем инструменты в Векторную БД
# Мы векторизуем Docstrings инструментов
tool_docs =[t.description for t in all_tools]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_texts(
    texts=tool_docs,
    embedding=embeddings,
    metadatas=[{"tool_name": t.name} for t in all_tools]
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # Берем только 1 самый подходящий инструмент!

/tmp/ipykernel_210/1842831375.py:31: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 📝 Задание 3: RAG для инструментов
Напишите функцию `execute_with_dynamic_tools(query: str)`.
1. Сделайте векторный поиск по `query` в `retriever`.
2. Извлеките имя найденного инструмента из `metadata`.
3. Найдите этот инструмент в списке `all_tools`.
4. Сделайте `llm.bind_tools([найденный_инструмент])` и вызовите модель с `query`.
5. Выведите, какой инструмент модель решила вызвать.

In [8]:
# TODO: Напишите логику динамической подгрузки инструментов
def execute_with_dynamic_tools(query: str):
    pass

# ТЕСТ
# execute_with_dynamic_tools("Пользователь забыл пароль, его логин ivanov_i. Помоги.")

<details>
<summary><b>💡 Показать решение (Задание 3)</b></summary>

```python
def execute_with_dynamic_tools(query: str):
    print(f"\n[USER QUERY]: {query}")
    
    # 1. Ищем релевантный инструмент в векторной БД
    docs = retriever.invoke(query)
    found_tool_name = docs[0].metadata["tool_name"]
    print(f"[VECTOR SEARCH]: Найден подходящий инструмент -> {found_tool_name}")
    
    # 2. Достаем сам объект инструмента из списка
    active_tool = next(t for t in all_tools if t.name == found_tool_name)
    
    # 3. Привязываем ТОЛЬКО этот инструмент к модели (экономим контекст!)
    llm_with_tool = llm.bind_tools([active_tool])
    
    # 4. Вызываем модель
    response = llm_with_tool.invoke(query)
    
    # 5. Проверяем Tool Calls
    if response.tool_calls:
        for tc in response.tool_calls:
            print(f"[LLM DECISION]: Модель вызывает {tc['name']} с аргументами {tc['args']}")
    else:
        print("[LLM DECISION]: Модель ответила текстом:", response.content)

# Тесты
execute_with_dynamic_tools("Пользователь забыл пароль, его логин ivanov_i. Помоги.")
execute_with_dynamic_tools("Упал микросервис auth-service в k8s, подними его.")
```
</details>

# Reliability Engineering

### 4. Обработка ошибок в Enterprise (Fallbacks & Retries) (Слайд 12)

В продакшене API отваливаются, а модели выдают 503 ошибки. В LCEL есть мощнейшие механизмы `.with_retry()` и `.with_fallbacks()`, которые позволяют строить каскадные системы (например, переключаться с падающей Qwen на резервную Llama).

In [9]:
from langchain_core.runnables import RunnableLambda

# Симулируем нестабильную облачную модель (которая падает в 100% случаев)
def unstable_llm_call(prompt: str):
    print("🌐 [Cloud LLM]: Пытаюсь сгенерировать ответ...")
    raise Exception("503 Service Unavailable (Cloud is down!)")

unstable_chain = RunnableLambda(unstable_llm_call)

# Надежная локальная модель
reliable_local_chain = (
    ChatPromptTemplate.from_template("Ответь на запрос: {query}")
    | llm
    | StrOutputParser()
)

### 📝 Задание 4: Каскадный Fallback
Соберите `enterprise_resilient_chain`.
1. Она должна попытаться выполнить `unstable_chain`.
2. Если `unstable_chain` падает, она должна автоматически использовать механизм `.with_fallbacks([reliable_local_chain])`.

In [10]:
# TODO: Примените .with_fallbacks
# enterprise_resilient_chain = ...

# print(enterprise_resilient_chain.invoke({"query": "Какая столица Франции?"}))

<details>
<summary><b>💡 Показать решение (Задание 4)</b></summary>

```python
# Настраиваем каскадный Fallback
# Если unstable_chain падает, LCEL автоматически перехватит ошибку
# и прозрачно для пользователя направит запрос в reliable_local_chain
enterprise_resilient_chain = unstable_chain.with_fallbacks([reliable_local_chain])

print("Запуск надежного пайплайна...")
result = enterprise_resilient_chain.invoke({"query": "Какая столица Франции?"})

print("\n[ФИНАЛЬНЫЙ ОТВЕТ]:")
print(result)

# P.S. В реальных проектах также используется .with_retry(stop_after_attempt=3)
# для повторных попыток перед переходом к fallback-модели.
```
</details>



### 🎉 Итоги Модуля 3
Вы освоили фундаментальные Enterprise-паттерны LangChain (без использования графов):
1. **Pydantic Guardrails**: Использование `Literal` и циклов самокоррекции для защиты от галлюцинаций.
2. **Semantic Routing**: Маршрутизация запросов для экономии ресурсов GPU через `RunnableLambda`.
3. **Tool Retrieval**: Векторный поиск инструментов (RAG для инструментов) для преодоления лимитов контекста.
4. **Reliability**: Использование `.with_fallbacks()` для обеспечения отказоустойчивости (High Availability) систем.

В следующем модуле мы перейдем к LangGraph, чтобы добавить этим цепочкам долговременную память (Persistence) и паттерны Human-in-the-Loop!